In [ ]:
import numpy as np
import pandas as pd
import os
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis

In [3]:
BASE_DATA_PATH = '../data/input_data/'
dataset_path = os.path.join('Borrow', 'Deposit')

print("Loading data...")
try:
    train_df = pd.read_csv(os.path.join(BASE_DATA_PATH, dataset_path, 'train.csv'))
except FileNotFoundError as e:
    print(f"Training data for {dataset_path} not found. Skipping. Details: {e}")

Loading data...


In [28]:
from utilities.preprocessing import preprocess
print("Preprocessing data...")
X_train, y_train, _ = preprocess(train_df)

Preprocessing data...
  - Dropping 3 zero-variance columns: ['userCoinTypeMode_Other', 'coinType_Other', 'reserve_Other']


In [29]:
# Convert X to float numpy array
X_train = np.asarray(X_train, dtype=float)

# Convert y to (time, event) float array
y_train = np.column_stack([
    train_df["timeDiff"].astype(float).values,
    train_df["status"].astype(int).values
])

In [31]:
from lassonet import LassoNetCoxRegressor

model = LassoNetCoxRegressor(tie_approximation="breslow")
model.fit(X_train, y_train)
# oracle, order, wrong, paths, prob = model.stability_selection(X_train, y_train)

OutOfMemoryError: CUDA out of memory. Tried to allocate 342.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 338.44 MiB is free. Process 3827274 has 516.00 MiB memory in use. Process 1382176 has 30.59 GiB memory in use. Including non-PyTorch memory, this process has 306.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter()

print("Training model on full training data...")
lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
        
model = CoxPHFitter(penalizer=0.1)
model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')

Training model on full training data...


In [ ]:
model.print_summary()

<lifelines.CoxPHFitter: fitted with 882736 total observations, 504835 right-censored observations>
             duration col = 'timeDiff'
                event col = 'status'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 882736
number of events observed = 377901
   partial log-likelihood = -4935950.25
         time fit was run = 2025-09-17 19:05:13 UTC

---
                                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                            
userActiveDaysWeekly                 0.18      1.19      0.00            0.17            0.18                1.19                1.20
marketLiquidationSumUSD             -0.02      0.98      0.00           -0.03           -0.02                0.97                0.98
marketLiquidationAvgAmountUSD        0.01      1.01      0.00            0.01            0.02                1.01                1.02
userBorrowSum                       -0.08      0.92      0.00           -0.09           -0.07                0.92                0.93
userDepositAvgAmountUSD              0.03      1.03      0.00            0.03            0.04                1.03                1.04
cosQuarter                           0.01      1.01      0.00            0.01            0.02                1.01                1.02
userWithdrawAvgAmountUSD            -0.10      0.91      0.00           -0.10           -0.09                0.90                0.91
userRepaySumUSD                     -0.09      0.91      0.00           -0.10           -0.09                0.90                0.92
userLiquidationAvgAmountUSD          0.02      1.02      0.00            0.01            0.02                1.01                1.02
userRepayCount                      -0.17      0.84      0.00           -0.18           -0.16                0.84                0.85
cosDayOfYear                         0.05      1.05      0.00            0.04            0.05                1.04                1.05
marketWithdrawSumUSD                -0.00      1.00      0.00           -0.01            0.00                0.99                1.00
userWithdrawAvgAmount               -0.05      0.95      0.00           -0.06           -0.05                0.94                0.95
marketRepayAvgAmount                -0.01      0.99      0.00           -0.02           -0.01                0.98                0.99
marketDepositCount                   0.00      1.00      0.00           -0.00            0.01                1.00                1.01
userBorrowAvgAmount                 -0.06      0.94      0.00           -0.06           -0.05                0.94                0.95
marketLiquidationCount               0.02      1.02      0.00            0.02            0.03                1.02                1.03
userSecondsSinceFirstTransaction     0.01      1.01      0.00            0.01            0.01                1.01                1.02
sinDayOfQuarter                     -0.02      0.98      0.00           -0.02           -0.01                0.98                0.99
userDepositCount                     0.21      1.24      0.00            0.21            0.22                1.23                1.24
quarter                             -0.00      1.00      0.00           -0.01            0.00                0.99                1.00
dayOfYear                           -0.00      1.00      0.00           -0.01            0.00                0.99                1.00
userRepayAvgAmount                   0.02      1.02      0.00            0.02            0.03                1.02                1.03
marketBorrowCount                    0.04      1.04      0.00            0.03            0.04                1.03                1.04
marketLiquidationAvgAmount           0.03      1.03      0.00            0

In [34]:
summary_df = model.summary
summary_df.head()

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
userActiveDaysWeekly,0.175140,1.191413,0.002160,0.170906,0.179374,1.186380,1.196468,0.0,81.081422,0.000000e+00,inf
marketLiquidationSumUSD,-0.022218,0.978027,0.002762,-0.027631,-0.016806,0.972747,0.983335,0.0,-8.045170,8.612584e-16,50.044403
marketLiquidationAvgAmountUSD,0.013923,1.014020,0.002371,0.009277,0.018569,1.009320,1.018743,0.0,5.873349,4.270789e-09,27.802850
userBorrowSum,-0.078968,0.924069,0.003110,-0.085064,-0.072873,0.918454,0.929719,0.0,-25.391962,3.093961e-142,470.084334
userDepositAvgAmountUSD,0.034029,1.034615,0.002282,0.029556,0.038502,1.029998,1.039253,0.0,14.911038,2.793753e-50,164.614200


In [ ]:
for index, row in summary_df.iterrows():
    if row['p'] >= 0.005:
        print(index)

marketWithdrawSumUSD
marketDepositCount
quarter
dayOfYear
cosDayOfWeek
priceInUSD
cosTimeOfDay
marketDepositSumUSD
marketDepositAvgAmount
isWeekend
userActiveDaysYearly
dayOfWeek


In [36]:
train_df.head()

,userActiveDaysWeekly,marketLiquidationSumUSD,marketLiquidationAvgAmountUSD,userBorrowSum,user,userDepositAvgAmountUSD,cosQuarter,userWithdrawAvgAmountUSD,userRepaySumUSD,userLiquidationAvgAmountUSD,...,amountUSD,userBorrowCount,cosDayOfQuarter,marketRepayCount,userBorrowSumUSD,userWithdrawSum,userDepositAvgAmount,userReserveMode,cosDayOfMonth,reserve
0,1.0,0.0,0.0,0.20000,0x4dee144e4d60ad8ae3e4b53e09669349dc0e23da,1.000000,6.123234e-17,0.0,0.0,0.0,...,0.199924,1,0.509933,0.0,0.199924,0.0,1.000000,DAI,-0.978148,other
1,1.0,0.0,0.0,0.79299,0x010afb8548a5d1a3a3d62f58ca0a5a1329974206,14.078625,6.123234e-17,0.0,0.0,0.0,...,0.793127,1,0.509933,0.0,0.793127,0.0,10.000000,DAI,-0.978148,other
2,1.0,0.0,0.0,1.00000,0x39d637737cc76c5849a52c7d3b872a1eb22aa71c,7.024463,6.123234e-17,0.0,0.0,0.0,...,0.999869,1,0.509933,0.0,0.999869,0.0,5.000000,USDC,-0.978148,USDC
3,1.0,0.0,0.0,15.00000,0x9f60699ce23f1ab86ec3e095b477ff79d4f409ad,27.100160,6.123234e-17,0.0,0.0,0.0,...,12.615414,3,0.509933,2.0,18.615467,0.0,26.500000,WPOL,-0.978148,WPOL
4,1.0,0.0,0.0,50.00000,0x005f16f017aa933bb41965b52848ceb8ee48b171,214.794320,6.123234e-17,0.0,0.0,0.0,...,55.015000,1,0.509933,2.0,55.015000,0.0,1.697614,jEUR,-0.978148,other


In [37]:
cols_list = ['marketWithdrawSumUSD', 'marketDepositCount', 'quarter', 'dayOfYear',
'cosDayOfWeek', 'priceInUSD', 'cosTimeOfDay', 'marketDepositSumUSD', 'marketDepositAvgAmount',
'isWeekend', 'userActiveDaysYearly', 'dayOfWeek']

In [38]:
new_train_df = train_df.drop(cols_list, axis=1)

In [ ]:
from utilities.preprocessing import preprocess
print("Preprocessing data...")
X_train, y_train, _ = preprocess(new_train_df)

Preprocessing data...
  - Dropping 3 zero-variance columns: ['userCoinTypeMode_Other', 'coinType_Other', 'reserve_Other']


In [40]:
from lifelines import CoxPHFitter

print("Training model on full training data...")
lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
        
model = CoxPHFitter(penalizer=0.1)
model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')

Training model on full training data...


<lifelines.CoxPHFitter: fitted with 882736 total observations, 504835 right-censored observations>

In [41]:
model.print_summary()

<lifelines.CoxPHFitter: fitted with 882736 total observations, 504835 right-censored observations>
             duration col = 'timeDiff'
                event col = 'status'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 882736
number of events observed = 377901
   partial log-likelihood = -4935958.99
         time fit was run = 2025-09-17 19:17:29 UTC

---
                                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                            
userActiveDaysWeekly                 0.18      1.19      0.00            0.17            0.18                1.19                1.20
marketLiquidationSumUSD             -0.02      0.98      0.00           -0.03           -0.02                0.97                0.98
marketLiquidationAvgAmountUSD        0.01      1.01      0.00            0.01            0.02                1.01                1.02
userBorrowSum                       -0.08      0.92      0.00           -0.08           -0.07                0.92                0.93
userDepositAvgAmountUSD              0.03      1.03      0.00            0.03            0.04                1.03                1.04
cosQuarter                           0.01      1.01      0.00            0.00            0.01                1.00                1.01
userWithdrawAvgAmountUSD            -0.10      0.91      0.00           -0.10           -0.09                0.90                0.91
userRepaySumUSD                     -0.09      0.91      0.00           -0.10           -0.09                0.90                0.92
userLiquidationAvgAmountUSD          0.02      1.02      0.00            0.01            0.02                1.01                1.02
userRepayCount                      -0.17      0.84      0.00           -0.18           -0.16                0.84                0.85
cosDayOfYear                         0.05      1.05      0.00            0.04            0.05                1.04                1.05
userWithdrawAvgAmount               -0.05      0.95      0.00           -0.06           -0.05                0.94                0.95
marketRepayAvgAmount                -0.01      0.99      0.00           -0.02           -0.01                0.98                0.99
userBorrowAvgAmount                 -0.06      0.94      0.00           -0.06           -0.05                0.94                0.95
marketLiquidationCount               0.02      1.02      0.00            0.02            0.03                1.02                1.03
userSecondsSinceFirstTransaction     0.01      1.01      0.00            0.01            0.02                1.01                1.02
sinDayOfQuarter                     -0.02      0.98      0.00           -0.02           -0.01                0.98                0.99
userDepositCount                     0.21      1.24      0.00            0.21            0.22                1.23                1.24
userRepayAvgAmount                   0.02      1.02      0.00            0.02            0.03                1.02                1.03
marketBorrowCount                    0.04      1.04      0.00            0.03            0.04                1.03                1.05
marketLiquidationAvgAmount           0.03      1.03      0.00            0.03            0.04                1.03                1.04
userWithdrawCount                    0.02      1.02      0.00            0.02            0.02                1.02                1.02
sinTimeOfDay                        -0.03      0.97      0.00           -0.03           -0.03                0.97                0.97
marketWithdrawCount                 -0.03      0.97      0.00           -0.04           -0.03                0.96                0.97
marketRepayAvgAmountUSD              0.03      1.03      0.00            0

In [42]:
# --- 4. Evaluate on Training Data ---
from utilities.cox_model import Model
from utilities.survival_metrics import get_concordance_index

print("Evaluating model on training data...")
wrapped_model_for_eval = Model(model)
train_preds = wrapped_model_for_eval.predict(X_train)
c_index_train = get_concordance_index(y_train, train_preds.values)
# training_scores[dataset_path] = c_index_train
print(f"  -> Training C-Index: {c_index_train:.4f}")

Evaluating model on training data...
  -> Training C-Index: 0.7407
